[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/41_cosine_lr_solution.ipynb)

# 🟢 Solution: Cosine LR Schedule with Warmup

*Training · Easy*

Reference implementation. Try it yourself in `41_cosine_lr.ipynb` first.

---
Implement the learning-rate schedule that essentially every modern transformer
is trained with: **linear warmup, then cosine decay**.

$$
\eta(t) =
\begin{cases}
\eta_{\max}\dfrac{t}{T_w} & t < T_w \\[2ex]
\eta_{\min} + \tfrac12(\eta_{\max}-\eta_{\min})
\left(1 + \cos\left(\pi\dfrac{t-T_w}{T-T_w}\right)\right) & t \ge T_w
\end{cases}
$$

### Signature
```python
def cosine_schedule(step, base_lr, warmup_steps, total_steps, min_lr=0.0):
    ...  # -> learning rate at `step`
```

### Rules
- `step` may be a **scalar or an array** of steps — the function must be
  vectorised, so branch with `jnp.where`, not `if`
- Must be `jax.jit`-able with `step` traced
- Clamp past the end: for `step >= total_steps` the rate stays at `min_lr`
- Do not use `optax.warmup_cosine_decay_schedule`

### Boundary conventions this is graded on
- $\eta(0) = 0$
- $\eta(T_w) = \eta_{\max}$ exactly — warmup ends *at* the peak
- $\eta(T) = \eta_{\min}$ exactly
- the halfway point of decay is $(\eta_{\max}+\eta_{\min})/2$

### Why warmup exists
At step 0 Adam's second-moment estimate $v$ is pure noise — it has seen exactly
one gradient — so $\hat{m}/\sqrt{\hat{v}}$ is an unreliable direction with
magnitude pinned near 1. Taking full-size steps in a badly-estimated direction
is how early training diverges, and the deeper the network the worse it is.
Warmup buys the moment estimates time to become meaningful. This is also why
[[adam]]'s bias correction and warmup are usually discussed together, and why
architectures that stabilise early gradients (pre-norm) need much less warmup.

Cosine decay then matters at the *other* end: it spends a long time at a high
rate and anneals smoothly to near zero, which empirically beats step decay and,
unlike a linear ramp to zero, does not waste the final steps at a rate too
small to make progress.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def cosine_schedule(step, base_lr, warmup_steps, total_steps, min_lr=0.0):
    step = jnp.asarray(step, dtype=jnp.float32)

    warmup_lr = base_lr * step / jnp.maximum(warmup_steps, 1)

    # Clip so the schedule flattens at min_lr instead of turning back upward
    # once step runs past total_steps.
    decay_span = jnp.maximum(total_steps - warmup_steps, 1)
    progress = jnp.clip((step - warmup_steps) / decay_span, 0.0, 1.0)
    cosine_lr = min_lr + 0.5 * (base_lr - min_lr) * (1 + jnp.cos(jnp.pi * progress))

    return jnp.where(step < warmup_steps, warmup_lr, cosine_lr)

In [ ]:
# 🔍 Verify
import jax.numpy as jnp

steps = jnp.arange(0, 1001, 100)
lrs = cosine_schedule(steps, base_lr=1e-3, warmup_steps=100, total_steps=1000)
for s, lr in zip(steps.tolist(), lrs.tolist()):
    bar = "█" * int(lr / 1e-3 * 40)
    print(f"{s:>5}  {lr:.6f}  {bar}")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("cosine_lr")